# Performing pseudobulking on the `InSituExperiment` object

In [1]:
## The following code ensures that all functions and init files are reloaded before executions.
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from insitupy import InSituData, InSituExperiment, CACHE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

### Load Xenium data into `InSituData` object

Now the Xenium data can be parsed by providing the data path to the `InSituPy` project folder.

In [4]:
insitupy_project = Path(CACHE / "out/demo_insitupy_project")
xd = InSituData.read(insitupy_project)
xd.load_all(skip="transcripts")
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project

    ➤ images
       CD20:	(25778, 35416)
       HE:	(25778, 35416, 3)
       HER2:	(25778, 35416)
       nuclei:	(25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           matrix
               AnnData object with n_obs × n_vars = 156447 × 297
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden', 'cell_type_dc', 'cell_type_dc_sub', 'cell_type_tacco', 'cell_type_publ_x', 'cell_type_publ_y', 'cell_type_dc_sub_final', 'cell_type_publ'
               var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
               uns: 'cell_type_dc_colors', 'cell_type_dc_sub', 'cell_type_dc_sub_colors', 'cell_type_dc_sub_final_colors', 'cell_type_p

### Create `InSituExperiment` from regions

In [5]:
exp = InSituExperiment.from_regions(
    data=xd, region_key="TMA"
)

A-1
A-2
A-3
B-1
B-2
B-3


In [6]:
exp

InSituExperiment with 6 samples:
           uid  CITAR slide_id    sample_id region_key region_name
0     40f223ea  ++-++  0001879  Replicate 1        TMA         A-1
1     53138ad1  ++-++  0001879  Replicate 1        TMA         A-2
2     5e2aff85  ++-++  0001879  Replicate 1        TMA         A-3
3     9f756276  ++-++  0001879  Replicate 1        TMA         B-1
4     360a9aab  ++-++  0001879  Replicate 1        TMA         B-2
5     c9a6cf2c  ++-++  0001879  Replicate 1        TMA         B-3

In [7]:
from insitupy.tools import generate_pseudobulk

In [8]:
psbulk = generate_pseudobulk(
    exp,
    sample_col='batch',
    groups_col='cell_type_dc_sub_final',
    counts_layer='counts',
    mode='sum',
    min_cells=10,
    min_counts=1000,
    skip_checks=False,
    min_prop=None,
    min_smpls=None,
    remove_empty=True
    )

In [9]:
psbulk

AnnData object with n_obs × n_vars = 60 × 297
    obs: 'batch', 'cell_type_dc_sub_final', 'psbulk_cells', 'psbulk_counts'
    layers: 'psbulk_props'

In [23]:
psbulk.obs

,batch,cell_type_dc_sub_final,psbulk_cells,psbulk_counts
9f756276-Adipocytes,9f756276,Adipocytes,11.0,2728.0
360a9aab-B cells,360a9aab,B cells,178.0,33777.0
40f223ea-B cells,40f223ea,B cells,86.0,12128.0
53138ad1-B cells,53138ad1,B cells,29.0,6606.0
5e2aff85-B cells,5e2aff85,B cells,67.0,13314.0
9f756276-B cells,9f756276,B cells,110.0,18702.0
c9a6cf2c-B cells,c9a6cf2c,B cells,184.0,30929.0
40f223ea-Breast cancer subtype 1,40f223ea,Breast cancer subtype 1,568.0,132467.0
53138ad1-Breast cancer subtype 1,53138ad1,Breast cancer subtype 1,967.0,213588.0
5e2aff85-Breast cancer subtype 1,5e2aff85,Breast cancer subtype 1,83.0,23719.0


In [10]:
psbulk.layers["psbulk_props"]

array([[0.36363636, 0.90909091, 0.72727273, ..., 0.36363636, 1.        ,
        0.45454545],
       [0.03932584, 0.56179775, 0.49438202, ..., 0.30898876, 0.68539326,
        0.21910112],
       [0.15116279, 0.54651163, 0.53488372, ..., 0.26744186, 0.56976744,
        0.22093023],
       ...,
       [0.11971831, 0.66197183, 0.71478873, ..., 0.3556338 , 0.60915493,
        0.25352113],
       [0.21165644, 0.66257669, 0.66871166, ..., 0.31288344, 0.52760736,
        0.23006135],
       [0.13095238, 0.49603175, 0.60714286, ..., 0.27380952, 0.51984127,
        0.19444444]], shape=(60, 297))

## Analyze pseudobulk data

The resulting pseudobulk `AnnData` object can be used for subsequent analyses using e.g. packages like `decoupler`. A tutorial on how to analyze pseudobulk data with `decoupler` can be found [here](https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html).